In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-07-03
Last modified on 2024-07-03
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, 
@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md
'''

'\nCreated on 2024-07-03\nLast modified on 2024-07-03\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, \n@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md\n'

#### **Requerimientos**

In [2]:
from stix2 import Filter, MemoryStore
import stix2
import requests

import os
import pandas as pd

import datetime
import csv

#### **Funciones**

In [3]:
def create_output_folder(path):
    '''
    Función encargada para crear el directorio facilitado en caso de no existir previamente.
    '''
    if not os.path.exists(path):
        os.makedirs(path)

In [4]:
def get_list_of_files_sub(dir_name):
    '''
    Función encargada de retornar una lista de archivos ubicados en la ruta facilitada así como en los subdirectorios disponibles.
    '''
    listOfFile = os.listdir(dir_name)
    allFiles = list()
    for entry in listOfFile:
        fullPath = os.path.join(dir_name, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + get_list_of_files_sub(fullPath)
        else:
            allFiles.append(fullPath)
    return allFiles

In [5]:
def get_unique_ttps_generated(paths):
    '''
    Función encargada de retornar una lista de id de TTP's únicas generadas. Esta función recibirá una ruta generada previamente en la estructura de carpetas de guardado. 
    '''
    sub_folders = set()
    for path in paths:
        dir, file = os.path.split(path)
        last_sub_folder = os.path.basename(dir).strip()
        sub_folders.add(last_sub_folder)
    return list(sub_folders)

In [6]:
def compare_lists(list1, list2):
    '''
    Función para comparar el contenido de dos listas.
    '''
    set1 = set(list1)
    set2 = set(list2)
    # Items en comun
    common_elements = set1.intersection(set2)
    # Items unicamente en list1
    unique_in_list1 = set1.difference(set2)
    # Items unicamente en list2
    unique_in_list2 = set2.difference(set1)
    # Items en común
    num_common_elements = len(common_elements)
    # Items no coincidentes
    num_non_common_elements = len(unique_in_list1) + len(unique_in_list2)
    return num_common_elements, list(common_elements), list(unique_in_list1), list(unique_in_list2), num_non_common_elements

In [7]:
def unique_list(series):
    '''
    Función para devolver sólo items únicos al agregar
    '''
    return list(set(series))

In [8]:
def get_data_from_branch(matrix):
    '''
    Función encargada de peticionar a la url de GitHub donde está publicada la última versión MITRE de la información en formato stix2. Retorna el objeto que contiene toda la información de MITRE.
    '''
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Verificar si la solicitud fue exitosa
        stix_json = response.json()
        
        if "objects" in stix_json:
            return MemoryStore(stix_data=stix_json["objects"])
        else:
            raise ValueError("JSON no contiene la clave 'objects'")
            
    except requests.RequestException as e:
        print(f"Error al obtener datos de {matrix}-attack: {e}")
        return None

In [9]:
def save_df_as_csv(df, name, aux_folder='stix2', aux_folder_2=''):
    now = datetime.datetime.now()
    if not os.path.exists(os.path.join(os.getcwd(), 'outputs', aux_folder, aux_folder_2)):
        os.makedirs(os.path.join(os.getcwd(), 'outputs', aux_folder,aux_folder_2))
    file_to_save = os.path.join(os.getcwd(), 'outputs', aux_folder, aux_folder_2, (name+ '_'+ now.strftime('%d%m%Y_%H%M')+'h.csv'))
    df.to_csv(file_to_save, sep=';', encoding='utf8', index=False, quoting=csv.QUOTE_NONNUMERIC)
    print('Archivo guardado correctamente '+ name + '_' +now.strftime('%d%m%Y_%H%M')+'h.csv')

In [10]:
def get_list_techniques_from_stix2(src, include="both"):
    '''
    Función encargada de la obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan tanto técnicas como subtécnicas. Retorna una lista de strings con el id correspondiente a cada técnica.
    '''
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques

In [11]:
def get_CP_and_NOCP_techniques_from_df(cp_techniques_list, df):
    '''
    Función encargada de filtrar un df dado el cual contiene el campo techniques_ID por las técnicas disponibles en CP (estas son facilitadas como parámetro en forma de lista). 
    '''
    cp_df = pd.DataFrame()
    nocp_df = pd.DataFrame() 
    try:
        cp_df = df[df['technique_ID'].isin(cp_techniques_list)]
        nocp_df = df[~df['technique_ID'].isin(cp_techniques_list)]
    except:
        print('No se ha podido ejecutar la operación.')
    return cp_df, nocp_df

In [12]:
def get_techniques_tactics_dataframe(matrix_store, cp_techniques_list):
    '''
    Función que, dada la matriz MITRE (entreprise, ics o mobile) devuelve un dataframe con la relación N:N, 1:N y N:1 entre técnicas y tácticas 
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    # Obtenemos las tácticas de la matriz
    tactics = matrix_store.query([
        Filter('type', '=', 'x-mitre-tactic')
    ])
    
    # Mediante diccionarios mapeamos la información que necesitamos
    technique_dict = {
        tech['external_references'][0]['external_id']: {
            'name': tech['name'],
            'deprecated': tech.get('x_mitre_deprecated', False),
            'revoked': tech.get('revoked', False)
        }
        for tech in techniques
        if 'external_references' in tech and len(tech['external_references']) > 0
    }
    tactic_dict = {
        tac['external_references'][0]['external_id']: tac['name']
        for tac in tactics
        if 'external_references' in tac and len(tac['external_references']) > 0
    }
    # Lista para almacenar las relaciones
    data = []

    # Recorrer todas las técnicas para encontrar sus relaciones con tácticas
    for technique in techniques:
        if 'kill_chain_phases' in technique:
            for phase in technique['kill_chain_phases']:
                # Cada fase representa una relación técnica-táctica
                tactic_shortname = phase['phase_name']
                external_id = technique['external_references'][0]['external_id'] if 'external_references' in technique and len(technique['external_references']) > 0 else None
                
                # Buscar el ID externo de la táctica correspondiente
                tactic_id = next((tac['external_references'][0]['external_id'] for tac in tactics if tac['x_mitre_shortname'] == tactic_shortname), None)
                # Si no hubiera ninguna relación
                if not external_id or not tactic_id:
                    continue
                # Agregamos la relación a la lista
                data.append({
                    "technique_ID": external_id,
                    "technique": technique_dict[external_id]['name'],
                    "tactic_ID": tactic_id,
                    "tactic": tactic_dict[tactic_id],
                    "technique_deprecated": technique_dict[external_id]['deprecated'],
                    "technique_revoked": technique_dict[external_id]['revoked']
                })
    
    # Creamos el dataframe final
    techniques_tactics_df = pd.DataFrame(data)
    #Filtramos técnicas deprecadas
    techniques_tactics_NN_df = techniques_tactics_df[(techniques_tactics_df['technique_deprecated']!=True)&(techniques_tactics_df['technique_revoked']!=True)]

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y tácticas en una relación N:N
    MITRE_techniques_tactics_NN_df = techniques_tactics_NN_df[['technique_ID', 'technique', 'tactic_ID', 'tactic']] # Filtramos las columnas que necesitamos
    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_tactics_NN_df)


    # Para finalizar agregamos por técnica y táctica con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y tácticas en una relación 1:N
    MITRE_technique_tactics_1N_df = MITRE_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    CP_technique_tactics_1N_df = CP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    NOCP_technique_tactics_1N_df = NOCP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })

    # Generamos el df compuesto por tecnicas y tácticas en una relación N:1
    MITRE_techniques_tactic_1N_df = MITRE_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_tactic_1N_df = CP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_tactic_1N_df = NOCP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_1N_df, CP_techniques_tactic_1N_df, NOCP_techniques_tactic_1N_df

In [13]:
def get_CP_ttps_with_rules(mitre_matrix, way='file'):
    '''
    Función encargada de obtner el listado de TTP's para las que desde CP disponemos de reglas de detección. Para la ejecutarla correctamente debe de haberse ejecutado previamente el código del bloque get_rules_and_classify_by_ttp. Hay dos formas de evaluar las técnicas disponibles y son gestionadas por el parámetro "way". Por defecto el modo es "file", lo que nos indica que se va a evaluar uno de los ficheros generados de resumen de reglas por TTP clasificadas. En el caso de que el parámetro "way" recoja el valor "path", lo que hará es identificar las reglas meciante la exploración de subdirectorios.  
    '''
    try:
        main_path = os.path.join(os.path.dirname(os.getcwd()), 'get_rules_and_classify_by_ttp', 'outputs', mitre_matrix)
        
        if not os.path.exists(main_path):
            raise ValueError(f'No existe la path: {main_path}')
        
        if way == 'file':
            file_path = os.path.join(main_path, f'{mitre_matrix}-ttp_all_classified_rules.csv')  # Corrección aquí
            file = pd.read_csv(file_path, sep=';')
            file = file[file['ttp'] != 'T0000']  # Filtramos la técnica ficticia donde metemos las reglas que no han sido mapeadas
            CP_techniques = file['ttp'].unique().tolist()  # Convertimos en lista de items únicos la columna que informa de la ttp
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)  # Ordenamos por longitud de caracteres para que cuando sea utilizada esta lista primero se evalúen las subtécnicas.

        elif way == 'path':
            CP_techniques = get_unique_ttps_generated(get_list_of_files_sub(main_path))
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)
            CP_techniques = [ttp for ttp in CP_techniques if ttp.startswith('T')]
            CP_techniques = [ttp for ttp in CP_techniques if ttp != 'T0000']

        else:
            CP_techniques = []
            
    except ValueError as e:
        print(f'{e}')
        CP_techniques = []
    
    return CP_techniques

#### **Funciones en desarrollo**

### **Parámetros**

In [14]:
matrix = 'enterprise'
save_as_csv = True
debug_df = True # Parámetro para controlar el printeado de df

### **Ejecución principal**

#### **1. Elementos generales**

##### **1.1 Generación de la matriz MITRE**

In [15]:
mitre_matrix = get_data_from_branch(matrix)
mitre_matrix

##### **1.2 Generación de la lista de técnicas (ID) de la matriz MITRE seleccionada**

In [16]:
techniques_list = get_list_techniques_from_stix2(mitre_matrix,'both')
print(f"Se han generado la lista de técnicas (ID) para la matriz {matrix.upper()} que contiene un total de {len(techniques_list)} TTP's")

Se han generado la lista de técnicas (ID) para la matriz ENTERPRISE que contiene un total de 637 TTP's


##### **1.3 Obtención de las TTP disponbles en Cyber Proof con regla de detección**

In [17]:
cp_techniques =  get_CP_ttps_with_rules(matrix, way='file')
print(f"Se han obtenido un total de {len(cp_techniques)} TTP's con regla de detección asociada disponibles en CP.")

Se han obtenido un total de 521 TTP's con regla de detección asociada disponibles en CP.


### **2. Relación técnicas - tácticas**

#### **Generación de las tablas**

In [18]:
MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_1N_df, CP_techniques_tactic_1N_df, NOCP_techniques_tactic_1N_df = get_techniques_tactics_dataframe(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [19]:
if debug_df:
    display(MITRE_techniques_tactics_NN_df.head(3))
    display(CP_techniques_tactics_NN_df.head(3))
    display(NOCP_techniques_tactics_NN_df.head(3))

,technique_ID,technique,tactic_ID,tactic
0,T1055.011,Extra Window Memory Injection,TA0005,Defense Evasion
1,T1055.011,Extra Window Memory Injection,TA0004,Privilege Escalation
2,T1053.005,Scheduled Task,TA0002,Execution


,technique_ID,technique,tactic_ID,tactic
0,T1055.011,Extra Window Memory Injection,TA0005,Defense Evasion
1,T1055.011,Extra Window Memory Injection,TA0004,Privilege Escalation
2,T1053.005,Scheduled Task,TA0002,Execution


,technique_ID,technique,tactic_ID,tactic
5,T1205.002,Socket Filters,TA0005,Defense Evasion
6,T1205.002,Socket Filters,TA0003,Persistence
7,T1205.002,Socket Filters,TA0011,Command and Control


In [20]:
if debug_df:
    display(MITRE_technique_tactics_1N_df.head(3))
    display(CP_technique_tactics_1N_df.head(3))
    display(NOCP_technique_tactics_1N_df.head(3))

,technique_ID,technique,tactic_ID,tactic
0,T1001,[Data Obfuscation],[TA0011],[Command and Control]
1,T1001.001,[Junk Data],[TA0011],[Command and Control]
2,T1001.002,[Steganography],[TA0011],[Command and Control]


,technique_ID,technique,tactic_ID,tactic
0,T1001,[Data Obfuscation],[TA0011],[Command and Control]
1,T1001.002,[Steganography],[TA0011],[Command and Control]
2,T1001.003,[Protocol Impersonation],[TA0011],[Command and Control]


,technique_ID,technique,tactic_ID,tactic
0,T1001.001,[Junk Data],[TA0011],[Command and Control]
1,T1011.001,[Exfiltration Over Bluetooth],[TA0010],[Exfiltration]
2,T1016.002,[Wi-Fi Discovery],[TA0007],[Discovery]


In [21]:
if debug_df:
    display(MITRE_techniques_tactic_1N_df.head(3))
    display(CP_techniques_tactic_1N_df.head(3))
    display(NOCP_techniques_tactic_1N_df.head(3))

,tactic_ID,tactic,technique_ID,technique
0,TA0001,[Initial Access],"[T1199, T1091, T1566.001, T1195, T1195.003, T1...","[Compromise Software Supply Chain, Compromise ..."
1,TA0002,[Execution],"[T1059.009, T1559.001, T1059.006, T1559.003, T...","[Inter-Process Communication, XPC Services, De..."
2,TA0003,[Persistence],"[T1205.002, T1546.016, T1543.002, T1556.008, T...","[Conditional Access Policies, Windows Manageme..."


,tactic_ID,tactic,technique_ID,technique
0,TA0001,[Initial Access],"[T1566.003, T1078.001, T1199, T1200, T1566, T1...","[Trusted Relationship, Drive-by Compromise, De..."
1,TA0002,[Execution],"[T1059.009, T1559.001, T1059.006, T1559, T1059...","[Inter-Process Communication, Deploy Container..."
2,TA0003,[Persistence],"[T1543.002, T1037.005, T1037, T1053.003, T1574...",[Windows Management Instrumentation Event Subs...


,tactic_ID,tactic,technique_ID,technique
0,TA0001,[Initial Access],"[T1566.004, T1659, T1195.003]","[Compromise Hardware Supply Chain, Spearphishi..."
1,TA0002,[Execution],"[T1651, T1059.008, T1559.003, T1059.010]","[AutoHotKey & AutoIT, Cloud Administration Com..."
2,TA0003,[Persistence],"[T1556.005, T1543.005, T1556.004, T1542.002, T...","[Additional Container Cluster Roles, Condition..."


#### **Guardado**

In [23]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_tactics_NN_df, f'[MITRE]_{matrix}_techniques_tactics_NN','stix2', 'Techniques_Tactics')
    save_df_as_csv(CP_techniques_tactics_NN_df, f'[CP]_{matrix}_techniques_tactics_NN','stix2', 'Techniques_Tactics')
    save_df_as_csv(NOCP_techniques_tactics_NN_df, f'[NOCP]_{matrix}_techniques_tactics_NN','stix2', 'Techniques_Tactics')

    save_df_as_csv(MITRE_technique_tactics_1N_df, f'[MITRE]_{matrix}_technique_tactics_1N','stix2', 'Techniques_Tactics')
    save_df_as_csv(CP_technique_tactics_1N_df, f'[CP]_{matrix}_technique_tactics_1N','stix2', 'Techniques_Tactics')
    save_df_as_csv(NOCP_technique_tactics_1N_df, f'[NOCP]_{matrix}_technique_tactics_1N','stix2', 'Techniques_Tactics')

    save_df_as_csv(MITRE_techniques_tactic_1N_df, f'[MITRE]_{matrix}_techniques_tactic_1N','stix2', 'Techniques_Tactics')
    save_df_as_csv(CP_techniques_tactic_1N_df, f'[CP]_{matrix}_techniques_tactic_1N','stix2', 'Techniques_Tactics')
    save_df_as_csv(NOCP_techniques_tactic_1N_df, f'[NOCP]_{matrix}_techniques_tactic_1N','stix2', 'Techniques_Tactics')

Archivo guardado correctamente [MITRE]_enterprise_techniques_tactics_NN_04072024_1236h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_tactics_NN_04072024_1236h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_tactics_NN_04072024_1236h.csv
Archivo guardado correctamente [MITRE]_enterprise_technique_tactics_1N_04072024_1236h.csv
Archivo guardado correctamente [CP]_enterprise_technique_tactics_1N_04072024_1236h.csv
Archivo guardado correctamente [NOCP]_enterprise_technique_tactics_1N_04072024_1236h.csv
Archivo guardado correctamente [MITRE]_enterprise_techniques_tactic_1N_04072024_1236h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_tactic_1N_04072024_1236h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_tactic_1N_04072024_1236h.csv


In [ ]:
CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df = get_CP_and_NOCP_techniques_from_df(CP_techniques, techniques_tactics_NN_df)

print(f'Tamaño del df original: {techniques_tactics_NN_df.shape}.')
print(f'Tamaño del df filtrado por técnicas diponibles en CP: {CP_techniques_tactics_NN_df.shape}.')
print(f'Tamaño del df filtrado por técnicas NO diponibles en CP: {NOCP_techniques_tactics_NN_df.shape}.')
display(CP_techniques_tactics_NN_df.head(2))
display(NOCP_techniques_tactics_NN_df.head(2))

Tamaño del df original: (830, 4).
Tamaño del df filtrado por técnicas diponibles en CP: (686, 4).
Tamaño del df filtrado por técnicas NO diponibles en CP: (144, 4).


,technique_ID,technique,tactic_ID,tactic
0,T1055.011,Extra Window Memory Injection,TA0005,Defense Evasion
1,T1055.011,Extra Window Memory Injection,TA0004,Privilege Escalation


,technique_ID,technique,tactic_ID,tactic
5,T1205.002,Socket Filters,TA0005,Defense Evasion
6,T1205.002,Socket Filters,TA0003,Persistence
